In [ ]:
### Step 1 — Setup

import sys, os
sys.path.insert(0, os.path.expanduser("~/Sleep_Stage_Research/conference"))

import json
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
from scipy.stats import wilcoxon

import config as cfg
from data_utils import (compute_class_weights, compute_channel_stats,
                        load_subjects_from_h5, load_participant_info)
from dataset import SleepSequenceDataset
from model import SleepStageNet
from trainer import Trainer
from eval_utils import (predict_on_subjects, compute_fold_metrics, aggregate_cv_results,
                        print_metrics, print_cv_summary, plot_confusion_matrix,
                        plot_training_history, save_results)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

h5_path = os.path.join(cfg.PREPROCESSED_DIR, "dreamt_psg7ch_epochs.h5")
pinfo = load_participant_info()

# AASM 4-level AHI classification
def assign_ahi_4level(ahi):
    if ahi < 5: return "Normal"
    elif ahi < 15: return "Mild"
    elif ahi < 30: return "Moderate"
    else: return "Severe"

pinfo["ahi_grp"] = pinfo["AHI"].apply(assign_ahi_4level)

AHI_GROUPS = ["Normal", "Mild", "Moderate", "Severe"]
ahi_groups = {}
for grp in AHI_GROUPS:
    subjects = sorted(pinfo[pinfo["ahi_grp"] == grp]["SID"].tolist())
    ahi_groups[grp] = subjects
    print(f"{grp}: n={len(subjects)}")

# Create fold assignments
NUM_FOLDS_41 = 5
ahi_folds = {}
for grp, subjects in ahi_groups.items():
    kf = KFold(n_splits=NUM_FOLDS_41, shuffle=True, random_state=cfg.SEED)
    folds = {}
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(subjects)):
        folds[fold_idx] = {"train": [subjects[i] for i in train_idx], "test": [subjects[i] for i in test_idx]}
    ahi_folds[grp] = folds

folds_path = os.path.join(cfg.CHECKPOINT_DIR, "exp4_1_aasm_fold_assignments.json")
with open(folds_path, "w") as f:
    json.dump({g: {str(k): v for k, v in folds.items()} for g, folds in ahi_folds.items()}, f, indent=2)
print(f"Fold assignments saved")

# Universal fold assignments for checkpoint selection
with open(os.path.join(cfg.CHECKPOINT_DIR, "fold_assignments.json")) as f:
    universal_folds = json.load(f)

# Load Exp 0 per-subject results
exp0_per_subj = {}
for fold_idx in range(cfg.NUM_FOLDS):
    rpath = os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{fold_idx}", "test_results.json")
    if os.path.exists(rpath):
        with open(rpath) as f:
            for sid, m in json.load(f)["per_subject"].items():
                exp0_per_subj[sid] = m
print(f"Loaded Exp 0 results: {len(exp0_per_subj)} subjects")

FT_LR = 1e-4
FT_MAX_EPOCHS = 20
FT_EARLY_STOP = 7
FT_LR_PATIENCE = 3
EXP_NAME = "exp4_1_aasm"
RNN_MODE = "bilstm"

In [ ]:
### Step 2 — FineTuneTrainer

class FineTuneTrainer(Trainer):
    def __init__(self, model, train_loader, val_loader, class_weights,
                 pretrained_path, exp_name, fold, device):
        super().__init__(model, train_loader, val_loader, class_weights,
                         exp_name=exp_name, fold=fold, device=device)
        self.optimizer = torch.optim.Adam(model.parameters(), lr=FT_LR)
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="min", patience=FT_LR_PATIENCE, factor=cfg.LR_FACTOR)
        self.pretrained_path = pretrained_path

    def train(self):
        if self.is_fold_complete():
            history_path = os.path.join(self.save_dir, "train_history.json")
            if os.path.exists(history_path):
                with open(history_path, "r") as f:
                    self.history = json.load(f)
            print(f"  [SKIP] {self.exp_name} fold {self.fold} already complete.")
            return self.history

        resumed = self.load_checkpoint()
        if not resumed:
            print(f"  [INIT] Loading pre-trained weights from {self.pretrained_path}")
            ckpt = torch.load(self.pretrained_path, map_location=self.device, weights_only=False)
            state = ckpt["model_state"]
            remapped = {k.replace("lstm.", "rnn.") if k.startswith("lstm.") else k: v for k, v in state.items()}
            self.model.load_state_dict(remapped)
            print(f"  [INIT] Pre-trained model loaded (val_loss={ckpt['best_val_loss']:.4f})")

        import time
        for epoch in range(self.start_epoch, FT_MAX_EPOCHS):
            t0 = time.time()
            current_lr = self.optimizer.param_groups[0]["lr"]
            print(f"\n  Epoch {epoch+1}/{FT_MAX_EPOCHS} | lr={current_lr:.2e}")

            train_loss, train_acc = self.train_one_epoch()
            val_loss, val_acc = self.validate()
            self.scheduler.step(val_loss)

            elapsed = time.time() - t0
            print(f"  => train_loss={train_loss:.4f} train_acc={train_acc:.3f} | "
                  f"val_loss={val_loss:.4f} val_acc={val_acc:.3f} | {elapsed:.0f}s")

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)
            self.history["lr"].append(current_lr)

            is_best = val_loss < self.best_val_loss
            if is_best:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                print(f"  ** New best val_loss: {val_loss:.4f}")
            else:
                self.patience_counter += 1
                print(f"  Patience: {self.patience_counter}/{FT_EARLY_STOP}")

            self.save_checkpoint(epoch, is_best=is_best)
            if self.patience_counter >= FT_EARLY_STOP:
                print(f"  [EARLY STOP] No improvement for {FT_EARLY_STOP} epochs.")
                break

        self.mark_complete()
        with open(os.path.join(self.save_dir, "train_history.json"), "w") as f:
            json.dump(self.history, f, indent=2)
        return self.history

def find_universal_checkpoint(test_subjects):
    min_overlap = len(test_subjects)
    best_fold = 0
    for fold_idx in range(cfg.NUM_FOLDS):
        exp0_test = set(universal_folds[str(fold_idx)]["test"])
        overlap = len(set(test_subjects) & exp0_test)
        if overlap == 0:
            path = os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{fold_idx}", "best_model.pt")
            if os.path.exists(path):
                return path, fold_idx
        if overlap < min_overlap:
            min_overlap = overlap
            best_fold = fold_idx
    return os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{best_fold}", "best_model.pt"), best_fold

print("FineTuneTrainer ready.")

In [ ]:
### Step 3 — Train all 4 AHI groups

all_results = {}
all_histories = {}

for grp in AHI_GROUPS:
    folds = ahi_folds[grp]
    all_results[grp] = []
    all_histories[grp] = []

    for fold_idx in range(NUM_FOLDS_41):
        exp_tag = f"{EXP_NAME}_{grp}_fold{fold_idx}"

        print(f"\n{'#'*70}")
        print(f"# FINE-TUNE {grp} (AHI) | FOLD {fold_idx+1}/{NUM_FOLDS_41}")
        print(f"{'#'*70}")

        train_subjects = folds[fold_idx]["train"]
        test_subjects = folds[fold_idx]["test"]

        np.random.seed(cfg.SEED + fold_idx)
        n_val = max(1, int(len(train_subjects) * cfg.VAL_RATIO))
        perm = np.random.permutation(len(train_subjects))
        val_subjects = [train_subjects[i] for i in perm[:n_val]]
        actual_train = [train_subjects[i] for i in perm[n_val:]]

        print(f"  Train: {len(actual_train)} | Val: {len(val_subjects)} | Test: {len(test_subjects)}")

        pretrained_path, univ_fold = find_universal_checkpoint(test_subjects)
        print(f"  Pre-trained from: exp0_fold{univ_fold}")

        stats = np.load(os.path.join(cfg.CHECKPOINT_DIR, f"exp0_fold{univ_fold}", "channel_stats.npz"))
        mean, std = stats["mean"], stats["std"]

        os.makedirs(os.path.join(cfg.CHECKPOINT_DIR, exp_tag), exist_ok=True)

        _, train_labels, _ = load_subjects_from_h5(h5_path, actual_train)
        class_weights = compute_class_weights(train_labels)
        del train_labels

        train_ds = SleepSequenceDataset(h5_path, actual_train, mean=mean, std=std)
        val_ds = SleepSequenceDataset(h5_path, val_subjects, mean=mean, std=std)
        print(f"  Train sequences: {len(train_ds)}, Val sequences: {len(val_ds)}")

        train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,
                                  num_workers=0, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, shuffle=False,
                                num_workers=0, pin_memory=True)

        model = SleepStageNet(n_channels=len(cfg.PSG_CHANNELS), rnn_mode=RNN_MODE).to(device)
        trainer = FineTuneTrainer(model, train_loader, val_loader, class_weights,
                                  pretrained_path=pretrained_path,
                                  exp_name=f"{EXP_NAME}_{grp}", fold=fold_idx, device=device)

        history = trainer.train()
        all_histories[grp].append(history)

        print(f"\n  [EVAL] Evaluating on {len(test_subjects)} test subjects...")
        trainer.load_best_model()
        results = predict_on_subjects(model, h5_path, test_subjects, mean, std, device=device)
        overall, per_subj = compute_fold_metrics(results)
        print_metrics(overall, title=f"FT-{grp} Fold {fold_idx}")
        all_results[grp].append((overall, per_subj))

        save_results({"overall": overall, "per_subject": {s: m for s, m in per_subj.items()}},
                     os.path.join(cfg.CHECKPOINT_DIR, exp_tag, "test_results.json"))

        del model, trainer, train_ds, val_ds, train_loader, val_loader
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'='*70}")
print(f"EXP 4.1 ALL AHI FINE-TUNING COMPLETE")
print(f"{'='*70}")

In [ ]:
### Step 4 — CV summaries + comparison

ahi_summaries = {}
ahi_per_subj = {}

for grp in AHI_GROUPS:
    summary = aggregate_cv_results(all_results[grp])
    ahi_summaries[grp] = summary
    print(f"\n{'='*60}")
    print(f"  FT-{grp} CV Summary")
    print(f"{'='*60}")
    print_cv_summary(summary)
    save_results(summary, os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_{grp}_cv_summary.json"))

    per_subj_all = {}
    for overall, per_subj in all_results[grp]:
        per_subj_all.update(per_subj)
    ahi_per_subj[grp] = per_subj_all

# Comparison vs universal
print(f"\n{'='*70}")
print(f"  COMPARISON: FT-AHI (AASM 4-level) vs Universal")
print(f"{'='*70}")

print(f"\n  {'Group':<12} {'n':>4} {'Universal κ':>14} {'Fine-Tuned κ':>14} {'Δκ':>8} {'Wilcoxon':>12}")
print(f"  {'-'*64}")

for grp in AHI_GROUPS:
    subjects = ahi_groups[grp]
    u_k = [exp0_per_subj[s]["kappa"] for s in subjects if s in exp0_per_subj]
    ft_k = [ahi_per_subj[grp][s]["kappa"] for s in subjects if s in ahi_per_subj[grp]]

    common = [s for s in subjects if s in exp0_per_subj and s in ahi_per_subj[grp]]
    p_str = "—"
    if len(common) > 5:
        stat, p = wilcoxon([exp0_per_subj[s]["kappa"] for s in common],
                           [ahi_per_subj[grp][s]["kappa"] for s in common])
        p_str = f"p={p:.4f}"

    delta = np.mean(ft_k) - np.mean(u_k)
    sign = "+" if delta > 0 else ""
    print(f"  {grp:<12} {len(subjects):>4} {np.mean(u_k):>10.4f}     {np.mean(ft_k):>10.4f}     {sign}{delta:.4f} {p_str:>12}")

save_results(ahi_summaries, os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_all_summaries.json"))

In [ ]:
### Step 5 — Visualizations

import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({"font.size": 11})

for grp in AHI_GROUPS:
    cm_total = np.zeros((cfg.NUM_CLASSES, cfg.NUM_CLASSES), dtype=np.int64)
    for overall, _ in all_results[grp]:
        cm_total += np.array(overall["confusion_matrix"])
    plot_confusion_matrix(cm_total, title=f"Exp 4.1: FT-{grp}",
                          save_path=os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_{grp}_confusion_matrix.png"))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(AHI_GROUPS))
width = 0.35

u_kappas = [np.mean([exp0_per_subj[s]["kappa"] for s in ahi_groups[g] if s in exp0_per_subj]) for g in AHI_GROUPS]
ft_kappas = [ahi_summaries[g]["kappa"]["mean"] for g in AHI_GROUPS]

bars1 = ax.bar(x - width/2, u_kappas, width, label="Universal", color="#3498db", edgecolor="black", linewidth=0.5)
bars2 = ax.bar(x + width/2, ft_kappas, width, label="Fine-Tuned", color="#2ecc71", edgecolor="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels([f"{g}\n(n={len(ahi_groups[g])})" for g in AHI_GROUPS])
ax.set_ylabel("Cohen's Kappa")
ax.set_title("Exp 4.1: Universal vs Fine-Tuned (AASM 4-level AHI)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{bar.get_height():.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(cfg.CHECKPOINT_DIR, f"{EXP_NAME}_comparison_chart.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nExperiment 4.1 complete.")